In [ ]:
!pip install pypdf
!pip install python-dotenv
!pip install transformers
!pip install llama-index
!pip install llama-index-llms-openai
!pip install llama-index-llms-replicate
!pip install llama-index-embeddings-huggingface
!pip install sentence-transformers
!pip install langchain
!pip install -U langchain-community
!pip install llama-index-embeddings-langchain


In [ ]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))


# Import the required classes from llama_index
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

In [ ]:
documents = SimpleDirectoryReader("/content/").load_data()
documents = [doc for doc in documents if doc.metadata['file_name'].endswith("pakistan_constitution.pdf")]

In [ ]:
!wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf


In [ ]:
!pip install llama-cpp-python

In [ ]:
import torch
from llama_cpp import Llama  # Import from llama-cpp-python

# Download the model file using wget or download it manually
model_path = '/content/mistral-7b-instruct-v0.1.Q4_K_M.gguf'  # Path to the downloaded model file

# Initialize the model using the local model path
llm2 = Llama(
    model_path=model_path,  # Provide the local path to the downloaded model
    temperature=0.1,
    max_tokens=256,
    context_window=3900,  # Allow some space for wiggle room within the context
    verbose=True
)

# Prepare the prompt for LLaMA model
prompt = "What are the executive powers of the Prime Minister according to the Pakistan Constitution?"

# Generate a response using the model
response = llm2(prompt)

# Display the response
print("Response:", response['choices'][0]['text'])


In [ ]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from llama_index.embeddings.langchain import LangchainEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import StorageContext

# Initialize HuggingFace embeddings
hf_embeddings = HuggingFaceEmbeddings(model_name="thenlper/gte-large")

# Wrap with LlamaIndex-compatible LangchainEmbedding
embed_model = LangchainEmbedding(hf_embeddings)

# Initialize document store and storage context
docstore = SimpleDocumentStore()
storage_context = StorageContext.from_defaults(docstore=docstore)

# Example documents
documents = documents

# Create the VectorStoreIndex using the documents, storage context, and embed model
index = VectorStoreIndex.from_documents(
    documents=documents,
    storage_context=storage_context,
    embed_model=embed_model
)

print("Index created successfully!")


In [ ]:
%pip install llama-index-llms-llama-cpp

In [ ]:
from llama_index.llms.llama_cpp import LlamaCPP
# Assuming llm2 is your Llama object from llama-cpp-python
llm = LlamaCPP(model_path=llm2.model_path) # Wrap llm2 into a LlamaCPP object

query_engine = index.as_query_engine(llm) # Pass the LlamaCPP object into as_query_engine
response = query_engine.query("What is the basic structure of the Pakistani government?")

In [ ]:
print(response)

In [ ]:
while True:
    query = input()
    if query.lower() == "exit":
        break
    response = query_engine.query(query)
    print(response)
